# Collaborative Comparison: LightGCN, NGCF and GraphSAGE

Notebook 09 showed the three graph models are implemented correctly and that LightGCN
reproduces its published Gowalla result. This notebook runs them on the dissertation
data using interactions only, no metadata. The single difference between these models
and the BPR-MF baseline from Notebook 08 is how embeddings are propagated over the
graph: same loss, same sampler, same optimiser, same dimension, same evaluator, same
splits. Anything that differs in the results is therefore attributable to propagation.

Tuning uses validation data and one seed. The selected configuration is then run across
five seeds and evaluated on test once per model and protocol.

Propagation depth is reported as a result rather than only tuned away. "Does adding
graph layers help when users hold seven interactions" is one of the questions the
dissertation is asking, so the whole depth curve is kept, not just its maximum.

## 1. Imports and configuration

In [1]:
%load_ext autoreload
%autoreload 2

import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow.parquet as pq
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src import graph as gr
from src import evaluation as ev
from src import models as md
from src import training as tr

# Paths
SUBSETS_DIR = PROJECT_ROOT / "data" / "processed" / "subsets_earliest"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# Experiment config
SUBSET = "small"
PROTOCOLS = ["last_out", "timestamp"]
K_VALUES = [10, 20]
EVAL_BATCH_USERS = 1024

SEEDS = [42, 1, 2, 3, 4]
SEED = SEEDS[0]                 # tuning uses one seed; only the final runs use all five

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Training config, matching Notebook 08 so the baseline stays comparable
DIM = 32                        # dimension had little effect at fixed regularisation
LR = 1e-3
BATCH = 8192
SAMPLING = "user"

TUNE_EPOCHS, TUNE_PATIENCE, TUNE_EVAL_EVERY = 150, 10, 5
FINAL_EPOCHS, FINAL_PATIENCE, FINAL_EVAL_EVERY = 300, 10, 5

# Search grids. A full layers x regularisation grid would be 96 runs; searching
# regularisation at a fixed depth and then sweeping depth is 42 and is the usual
# compromise. Depth and regularisation do interact, since propagation smooths
# embeddings and is itself a form of regularisation, so this is a limitation to state.
REG_GRID = [0.05, 0.10, 0.15, 0.20]
LAYER_GRID = [1, 2, 3, 4]
REG_SEARCH_LAYERS = 3           # depth held here while regularisation is searched

MODELS = {
    "LightGCN":  {"class": md.LightGCN,  "adjacency": "symmetric", "layers": 3},
    "NGCF":      {"class": md.NGCF,      "adjacency": "symmetric", "layers": 3},
    "GraphSAGE": {"class": md.GraphSAGE, "adjacency": "mean",      "layers": 2},
}

# Notebook 08 results, carried forward as the comparison point
BPR_MF = {"dim": 32, "l2_reg": {"last_out": 0.10, "timestamp": 0.15},
          "ndcg@10": {"last_out": 0.00693, "timestamp": 0.00497}}

torch.manual_seed(SEED)
np.random.seed(SEED)

n_tuning = len(MODELS) * len(PROTOCOLS) * (len(REG_GRID) + len(LAYER_GRID) - 1)
n_final = len(MODELS) * len(PROTOCOLS) * len(SEEDS)

print(f"device: {DEVICE}")
print(f"subset: {SUBSET}, protocols: {PROTOCOLS}, seeds: {SEEDS}")
print(f"dim {DIM}, lr {LR}, batch {BATCH}, sampling {SAMPLING}")
print(f"\ntuning runs: {n_tuning}   final runs: {n_final}")

device: cuda
subset: small, protocols: ['last_out', 'timestamp'], seeds: [42, 1, 2, 3, 4]
dim 32, lr 0.001, batch 8192, sampling user

tuning runs: 42   final runs: 30


## 2. Data, graphs and evaluators

Each protocol has its own training set, so each needs its own graph. Only training
interactions become edges - validation and test must not be visible to propagation,
which is checked below rather than assumed.

Two adjacency matrices per protocol: the symmetric normalisation for LightGCN and NGCF,
and the row-normalised one for GraphSAGE's mean aggregation. Both are stored in CSR
format, which halved training time in Notebook 09.

Three evaluators per protocol. Test is for reported results and is used once per model.
Validation drives early stopping. A 10,000-user subset of validation drives the grid
search, because 42 tuning runs each evaluating 50,000 users would spend a large fraction
of the budget on evaluation rather than training. Under the timestamp protocol only
users who actually have a validation interaction are used for selection, since the
validation window is a date range and roughly half the evaluation users fall outside it.

In [2]:
def load_protocol(subset, protocol):
    base = SUBSETS_DIR / subset
    splits = base / "splits" / protocol
    parts = {
        name: pq.read_table(splits / f"{name}.parquet",
                            columns=["user_idx", "item_idx"]).to_pandas()
        for name in ["train", "validation", "test"]
    }
    return {
        "protocol": protocol,
        "n_users": len(pq.read_table(base / "user_mapping.parquet")),
        "n_items": len(pq.read_table(base / "item_mapping.parquet")),
        "eval_users": pq.read_table(
            splits / "eval_users.parquet").to_pandas()["user_idx"].to_numpy(),
        "cold_items": pq.read_table(
            splits / "cold_items.parquet").to_pandas()["item_idx"].to_numpy(),
        **parts,
    }


data = {p: load_protocol(SUBSET, p) for p in PROTOCOLS}

pd.DataFrame([{
    "protocol": p,
    "users": d["n_users"], "items": d["n_items"],
    "train": len(d["train"]), "validation": len(d["validation"]), "test": len(d["test"]),
    "eval_users": len(d["eval_users"]),
} for p, d in data.items()])

,protocol,users,items,train,validation,test,eval_users
0,last_out,258597,105298,1829582,258597,258597,50000
1,timestamp,258597,105298,2020041,165504,161231,50000


In [3]:
graphs = {}
for protocol, d in data.items():
    n_nodes = d["n_users"] + d["n_items"]
    edge_index = gr.build_edge_index(d["train"], d["n_users"], d["n_items"])

    symmetric = gr.normalised_adjacency(edge_index, n_nodes, device=DEVICE)
    radius, iterations = gr.spectral_radius(symmetric)

    # held-out interactions must not appear as edges, otherwise propagation sees the
    # answer. This is the one leak that would be invisible in the metrics.
    leaked = (gr.count_held_out_edges(edge_index, d["validation"], d["n_users"])
              + gr.count_held_out_edges(edge_index, d["test"], d["n_users"]))

    graphs[protocol] = {
        "edge_index": edge_index,
        "n_nodes": n_nodes,
        "symmetric": symmetric.to_sparse_csr(),
        "mean": gr.mean_adjacency(edge_index, n_nodes, device=DEVICE).to_sparse_csr(),
    }

    print(f"{protocol}: {edge_index.shape[1] // 2:,} edges over {n_nodes:,} nodes | "
          f"symmetric {gr.verify_symmetry(edge_index)} | "
          f"spectral radius {radius:.6f} ({iterations} it) | held-out edges {leaked}")

    assert leaked == 0
    assert abs(radius - 1.0) < 1e-3

c:\Projects\gnn-recommender-dissertation\src\graph.py:78: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\Context.cpp:767.)
  adj = torch.sparse_coo_tensor(
C:\Users\xhani\AppData\Local\Temp\ipykernel_43500\3922316726.py:17: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\SparseCsrTensorImpl.cpp:51.)
  "symmetric": symmetric.to_sparse_csr(),


last_out: 1,829,582 edges over 363,895 nodes | symmetric True | spectral radius 1.000000 (43 it) | held-out edges 0
timestamp: 2,020,041 edges over 363,895 nodes | symmetric True | spectral radius 1.000000 (31 it) | held-out edges 0


In [4]:
test_evaluators, val_evaluators, tune_evaluators = {}, {}, {}

for protocol, d in data.items():
    test_evaluators[protocol] = ev.Evaluator.from_frames(
        ground_truth=d["test"],
        mask_frames=[d["train"], d["validation"]],
        eval_users=d["eval_users"],
        n_users=d["n_users"], n_items=d["n_items"],
        k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
    )

    # restrict selection to users who actually have a validation interaction.
    # under leave-last-out that is everyone; under timestamp it is roughly half.
    probe = ev.Evaluator.from_frames(
        ground_truth=d["validation"], mask_frames=[d["train"]],
        eval_users=d["eval_users"], n_users=d["n_users"], n_items=d["n_items"],
        k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
    )
    usable = d["eval_users"][np.diff(probe.truth_indptr)[d["eval_users"]] > 0]

    for store, users in [(val_evaluators, usable), (tune_evaluators, usable[:10_000])]:
        store[protocol] = ev.Evaluator.from_frames(
            ground_truth=d["validation"], mask_frames=[d["train"]],
            eval_users=users, n_users=d["n_users"], n_items=d["n_items"],
            k_values=K_VALUES, device=DEVICE, batch_users=EVAL_BATCH_USERS,
        )

    print(f"{protocol}: test {len(d['eval_users']):,} users | "
          f"validation {len(usable):,} | tuning {len(tune_evaluators[protocol].eval_users):,}")

last_out: test 50,000 users | validation 50,000 | tuning 10,000
timestamp: test 50,000 users | validation 24,514 | tuning 10,000


In [5]:
def build_model(name, protocol, n_layers, seed=SEED, **overrides):
    spec = MODELS[name]
    d = data[protocol]
    return spec["class"](
        d["n_users"], d["n_items"], dim=DIM, seed=seed,
        adjacency=graphs[protocol][spec["adjacency"]],
        n_layers=n_layers, **overrides,
    )


def run(name, protocol, n_layers, l2_reg, seed=SEED, evaluator=None,
        epochs=TUNE_EPOCHS, patience=TUNE_PATIENCE, eval_every=TUNE_EVAL_EVERY,
        verbose=False, **overrides):
    """Train one configuration. Defaults are the tuning settings."""
    return tr.train_bpr(
        build_model(name, protocol, n_layers, seed, **overrides),
        data[protocol]["train"], data[protocol]["n_items"],
        evaluator=evaluator if evaluator is not None else tune_evaluators[protocol],
        epochs=epochs, batch_size=BATCH, learning_rate=LR, l2_reg=l2_reg,
        sampling=SAMPLING, patience=patience, eval_every=eval_every,
        selection_metric="ndcg@10", device=DEVICE, seed=seed, verbose=verbose,
    )


# time one configuration of each model before committing to 72 runs
for name in MODELS:
    probe = run(name, "last_out", MODELS[name]["layers"], 0.10, epochs=3, evaluator=None)
    per_epoch = probe["history"]["seconds"].mean()
    print(f"{name:>10}: {per_epoch:.2f}s per epoch -> "
          f"{per_epoch * TUNE_EPOCHS / 60:.0f} min per tuning run")

c:\Projects\gnn-recommender-dissertation\src\training.py:65: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  users = torch.as_tensor(train["user_idx"].to_numpy(), dtype=torch.long, device=device)
c:\Projects\gnn-recommender-dissertation\src\training.py:128: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\autograd\generated\python_variable_methods.cpp:837.)
  epoch_loss += float(loss) * len(batch)


  LightGCN: 7.06s per epoch -> 18 min per tuning run
      NGCF: 11.30s per epoch -> 28 min per tuning run
 GraphSAGE: 6.95s per epoch -> 17 min per tuning run


In [6]:
convergence = run("LightGCN", "last_out", n_layers=3, l2_reg=0.10,
                  epochs=150, patience=10 ** 6, eval_every=5, verbose=True)

  epoch   5  loss 0.5093  val ndcg@10 0.00967  *
  epoch  10  loss 0.4849  val ndcg@10 0.00966
  epoch  15  loss 0.4832  val ndcg@10 0.00965
  epoch  20  loss 0.4832  val ndcg@10 0.00966
  epoch  25  loss 0.4834  val ndcg@10 0.00966
  epoch  30  loss 0.4832  val ndcg@10 0.00966
  epoch  35  loss 0.4833  val ndcg@10 0.00966
  epoch  40  loss 0.4833  val ndcg@10 0.00965
  epoch  45  loss 0.4833  val ndcg@10 0.00965
  epoch  50  loss 0.4832  val ndcg@10 0.00965
  epoch  55  loss 0.4831  val ndcg@10 0.00965
  epoch  60  loss 0.4830  val ndcg@10 0.00965
  epoch  65  loss 0.4829  val ndcg@10 0.00967  *
  epoch  70  loss 0.4830  val ndcg@10 0.00965
  epoch  75  loss 0.4830  val ndcg@10 0.00962
  epoch  80  loss 0.4833  val ndcg@10 0.00965
  epoch  85  loss 0.4831  val ndcg@10 0.00962
  epoch  90  loss 0.4830  val ndcg@10 0.00965
  epoch  95  loss 0.4828  val ndcg@10 0.00966
  epoch 100  loss 0.4831  val ndcg@10 0.00966
  epoch 105  loss 0.4833  val ndcg@10 0.00962
  epoch 110  loss 0.4836  va

In [6]:
from src import baselines as bl

pop_reference = {}
for protocol, d in data.items():
    pop = bl.PopularityRecommender().fit(d["train"], d["n_items"])
    pop_reference[protocol] = tune_evaluators[protocol].evaluate(
        pop.score_fn(DEVICE), return_per_user=False)["means"]["ndcg@10"]
    print(f"{protocol}: Popularity ndcg@10 on the tuning subset {pop_reference[protocol]:.5f}")

last_out: Popularity ndcg@10 on the tuning subset 0.00998
timestamp: Popularity ndcg@10 on the tuning subset 0.00390


In [9]:
coarse = []
for reg in [1e-4, 1e-3, 1e-2, 5e-2, 1e-1]:
    outcome = run("LightGCN", "last_out", n_layers=3, l2_reg=reg,
                  epochs=60, patience=8, eval_every=5)
    h = outcome["history"]
    coarse.append({
        "l2_reg": reg,
        "best_epoch": outcome["best_epoch"],
        "val_ndcg@10": outcome["best_val_score"],
        "final_loss": h["loss"].iloc[-1],
        "beats_popularity": outcome["best_val_score"] > pop_reference["last_out"],
    })
    print(coarse[-1], flush=True)

pd.DataFrame(coarse)

{'l2_reg': 0.0001, 'best_epoch': 50, 'val_ndcg@10': 0.01165313832461834, 'final_loss': np.float64(0.055931847912169784), 'beats_popularity': True}
{'l2_reg': 0.001, 'best_epoch': 60, 'val_ndcg@10': 0.011379247531294823, 'final_loss': np.float64(0.06971465828046548), 'beats_popularity': True}
{'l2_reg': 0.01, 'best_epoch': 55, 'val_ndcg@10': 0.010864912532269955, 'final_loss': np.float64(0.17943623190582256), 'beats_popularity': True}
{'l2_reg': 0.05, 'best_epoch': 30, 'val_ndcg@10': 0.01001669466495514, 'final_loss': np.float64(0.3898819249064641), 'beats_popularity': True}
{'l2_reg': 0.1, 'best_epoch': 5, 'val_ndcg@10': 0.009668715298175812, 'final_loss': np.float64(0.483286684298275), 'beats_popularity': False}


,l2_reg,best_epoch,val_ndcg@10,final_loss,beats_popularity
0,0.0001,50,0.011653,0.055932,True
1,0.0010,60,0.011379,0.069715,True
2,0.0100,55,0.010865,0.179436,True
3,0.0500,30,0.010017,0.389882,True
4,0.1000,5,0.009669,0.483287,False


In [10]:
coarse2 = []
for reg in [1e-5, 5e-5, 1e-4, 3e-4]:
    outcome = run("LightGCN", "last_out", n_layers=3, l2_reg=reg,
                  epochs=150, patience=12, eval_every=5)
    coarse2.append({
        "l2_reg": reg,
        "best_epoch": outcome["best_epoch"],
        "val_ndcg@10": outcome["best_val_score"],
        "final_loss": outcome["history"]["loss"].iloc[-1],
        "epochs_run": len(outcome["history"]),
    })
    print(coarse2[-1], flush=True)

pd.DataFrame(coarse2)

{'l2_reg': 1e-05, 'best_epoch': 85, 'val_ndcg@10': 0.011762718670070171, 'final_loss': np.float64(0.01553863153531592), 'epochs_run': 145}
{'l2_reg': 5e-05, 'best_epoch': 65, 'val_ndcg@10': 0.011734841391444206, 'final_loss': np.float64(0.019194581151173923), 'epochs_run': 125}
{'l2_reg': 0.0001, 'best_epoch': 100, 'val_ndcg@10': 0.011657113209366798, 'final_loss': np.float64(0.016133358898866947), 'epochs_run': 150}
{'l2_reg': 0.0003, 'best_epoch': 120, 'val_ndcg@10': 0.011861531063914299, 'final_loss': np.float64(0.01972512439216185), 'epochs_run': 150}


,l2_reg,best_epoch,val_ndcg@10,final_loss,epochs_run
0,0.00001,85,0.011763,0.015539,145
1,0.00005,65,0.011735,0.019195,125
2,0.00010,100,0.011657,0.016133,150
3,0.00030,120,0.011862,0.019725,150


In [11]:
converge = {}
for reg in [3e-4, 1e-3]:
    print(f"reg {reg}")
    outcome = run("LightGCN", "last_out", n_layers=3, l2_reg=reg,
                  epochs=300, patience=20, eval_every=10, verbose=True)
    converge[reg] = outcome
    print(f"  best epoch {outcome['best_epoch']}, "
          f"val ndcg@10 {outcome['best_val_score']:.5f}, "
          f"epochs run {len(outcome['history'])}\n")

reg 0.0003
  epoch  10  loss 0.3181  val ndcg@10 0.01009  *
  epoch  20  loss 0.2095  val ndcg@10 0.01047  *
  epoch  30  loss 0.1440  val ndcg@10 0.01077  *
  epoch  40  loss 0.1027  val ndcg@10 0.01141  *
  epoch  50  loss 0.0759  val ndcg@10 0.01153  *
  epoch  60  loss 0.0590  val ndcg@10 0.01169  *
  epoch  70  loss 0.0473  val ndcg@10 0.01146
  epoch  80  loss 0.0396  val ndcg@10 0.01144
  epoch  90  loss 0.0340  val ndcg@10 0.01140
  epoch 100  loss 0.0299  val ndcg@10 0.01148
  epoch 110  loss 0.0269  val ndcg@10 0.01164
  epoch 120  loss 0.0244  val ndcg@10 0.01186  *
  epoch 130  loss 0.0225  val ndcg@10 0.01181
  epoch 140  loss 0.0212  val ndcg@10 0.01156
  epoch 150  loss 0.0197  val ndcg@10 0.01171
  epoch 160  loss 0.0190  val ndcg@10 0.01157
  epoch 170  loss 0.0181  val ndcg@10 0.01147
  epoch 180  loss 0.0174  val ndcg@10 0.01164
  epoch 190  loss 0.0169  val ndcg@10 0.01135
  epoch 200  loss 0.0164  val ndcg@10 0.01126
  epoch 210  loss 0.0157  val ndcg@10 0.01122
  

In [12]:
lr_probe = tr.train_bpr(
    build_model("LightGCN", "last_out", n_layers=3, seed=SEED),
    data["last_out"]["train"], data["last_out"]["n_items"],
    evaluator=tune_evaluators["last_out"],
    epochs=150, batch_size=BATCH, learning_rate=3e-3, l2_reg=1e-3,
    sampling=SAMPLING, patience=10 ** 6, eval_every=10,
    selection_metric="ndcg@10", device=DEVICE, seed=SEED, verbose=True,
)
print(f"\nlr 3e-3: best epoch {lr_probe['best_epoch']}, "
      f"val ndcg@10 {lr_probe['best_val_score']:.5f}")

  epoch  10  loss 0.1525  val ndcg@10 0.01050  *
  epoch  20  loss 0.0711  val ndcg@10 0.01131  *
  epoch  30  loss 0.0465  val ndcg@10 0.01161  *
  epoch  40  loss 0.0369  val ndcg@10 0.01136
  epoch  50  loss 0.0320  val ndcg@10 0.01145
  epoch  60  loss 0.0294  val ndcg@10 0.01149
  epoch  70  loss 0.0273  val ndcg@10 0.01161
  epoch  80  loss 0.0263  val ndcg@10 0.01206  *
  epoch  90  loss 0.0252  val ndcg@10 0.01169
  epoch 100  loss 0.0245  val ndcg@10 0.01208  *
  epoch 110  loss 0.0240  val ndcg@10 0.01169
  epoch 120  loss 0.0235  val ndcg@10 0.01185
  epoch 130  loss 0.0231  val ndcg@10 0.01200
  epoch 140  loss 0.0230  val ndcg@10 0.01208  *
  epoch 150  loss 0.0225  val ndcg@10 0.01234  *

lr 3e-3: best epoch 150, val ndcg@10 0.01234


In [7]:
TUNING_CACHE = RESULTS_DIR / "10_tuning"
TUNING_CACHE.mkdir(exist_ok=True)


def tune_one(name, protocol, n_layers, l2_reg, epochs=300, seed=SEED, **overrides):
    """Run one configuration, or return the cached result if it has already been run.

    Histories are written as soon as a run finishes, so an interrupted session only
    loses the run that was in progress.
    """
    extra = "".join(f"_{k}{v}" for k, v in sorted(overrides.items()))
    tag = f"{name}_{protocol}_L{n_layers}_reg{l2_reg:g}_e{epochs}{extra}_s{seed}"
    path = TUNING_CACHE / f"{tag}.csv"

    if path.exists():
        history, cached = pd.read_csv(path), True
    else:
        t0 = time.time()
        outcome = run(name, protocol, n_layers, l2_reg, seed=seed, epochs=epochs,
                      patience=10, eval_every=10, **overrides)
        history, cached = outcome["history"], False
        history.to_csv(path, index=False)
        print(f"  {tag}: {(time.time() - t0) / 60:.1f} min")

    evaluated = history.dropna(subset=["val_ndcg@10"])
    best = evaluated.loc[evaluated["val_ndcg@10"].idxmax()]
    return {"model": name, "protocol": protocol, "n_layers": n_layers,
            "l2_reg": l2_reg, "seed": seed, **overrides,
            "best_epoch": int(best["epoch"]),
            "val_ndcg@10": float(best["val_ndcg@10"]),
            "epochs_run": len(history), "cached": cached}

In [14]:
# saving the 300 epochs runs already done

for reg, outcome in converge.items():
    outcome["history"].to_csv(
        TUNING_CACHE / f"LightGCN_last_out_L3_reg{reg:g}_s{SEED}.csv", index=False)
print(sorted(p.name for p in TUNING_CACHE.glob("*.csv")))

['LightGCN_last_out_L3_reg0.0003_s42.csv', 'LightGCN_last_out_L3_reg0.001_s42.csv']


In [15]:
phase_a_lightgcn = pd.DataFrame([
    tune_one("LightGCN", "last_out", n_layers=3, l2_reg=reg)
    for reg in [1e-3, 3e-3, 1e-2]
])
phase_a_lightgcn

  LightGCN_last_out_L3_reg0.003_s42: 32.3 min
  LightGCN_last_out_L3_reg0.01_s42: 30.2 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,LightGCN,last_out,3,0.001,42,230,0.012407,300,True
1,LightGCN,last_out,3,0.003,42,300,0.012740,300,False
2,LightGCN,last_out,3,0.010,42,280,0.011910,300,False


In [17]:
budget_probe = run("LightGCN", "last_out", n_layers=3, l2_reg=3e-3,
                   epochs=500, patience=10 ** 6, eval_every=10, verbose=True)
h = budget_probe["history"].dropna(subset=["val_ndcg@10"])
print(f"\npeak {h['val_ndcg@10'].max():.5f} at epoch "
      f"{int(h.loc[h['val_ndcg@10'].idxmax(), 'epoch'])}")

  epoch  10  loss 0.3257  val ndcg@10 0.00999  *
  epoch  20  loss 0.2327  val ndcg@10 0.01029  *
  epoch  30  loss 0.1763  val ndcg@10 0.01054  *
  epoch  40  loss 0.1394  val ndcg@10 0.01061  *
  epoch  50  loss 0.1144  val ndcg@10 0.01095  *
  epoch  60  loss 0.0983  val ndcg@10 0.01096  *
  epoch  70  loss 0.0871  val ndcg@10 0.01113  *
  epoch  80  loss 0.0797  val ndcg@10 0.01120  *
  epoch  90  loss 0.0740  val ndcg@10 0.01123  *
  epoch 100  loss 0.0697  val ndcg@10 0.01130  *
  epoch 110  loss 0.0667  val ndcg@10 0.01142  *
  epoch 120  loss 0.0639  val ndcg@10 0.01159  *
  epoch 130  loss 0.0616  val ndcg@10 0.01187  *
  epoch 140  loss 0.0602  val ndcg@10 0.01166
  epoch 150  loss 0.0585  val ndcg@10 0.01179
  epoch 160  loss 0.0576  val ndcg@10 0.01187
  epoch 170  loss 0.0567  val ndcg@10 0.01200  *
  epoch 180  loss 0.0558  val ndcg@10 0.01218  *
  epoch 190  loss 0.0552  val ndcg@10 0.01234  *
  epoch 200  loss 0.0545  val ndcg@10 0.01234  *
  epoch 210  loss 0.0536  val

In [18]:
for path in TUNING_CACHE.glob("*_s42.csv"):
    if "_e" not in path.name:
        path.rename(path.with_name(path.name.replace("_s42", "_e300_s42")))
print(sorted(p.name for p in TUNING_CACHE.glob("*.csv")))

['LightGCN_last_out_L3_reg0.0003_e300_s42.csv', 'LightGCN_last_out_L3_reg0.001_e300_s42.csv', 'LightGCN_last_out_L3_reg0.003_e300_s42.csv', 'LightGCN_last_out_L3_reg0.01_e300_s42.csv']


In [19]:
phase_b = pd.DataFrame([
    tune_one("LightGCN", "last_out", n_layers=L, l2_reg=3e-3, epochs=300)
    for L in [1, 2, 3, 4]
])
phase_b

  LightGCN_last_out_L1_reg0.003_e300_s42: 6.7 min
  LightGCN_last_out_L2_reg0.003_e300_s42: 20.8 min
  LightGCN_last_out_L4_reg0.003_e300_s42: 37.5 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,LightGCN,last_out,1,0.003,42,40,0.010865,140,False
1,LightGCN,last_out,2,0.003,42,300,0.012128,300,False
2,LightGCN,last_out,3,0.003,42,300,0.012740,300,True
3,LightGCN,last_out,4,0.003,42,290,0.012107,300,False


In [20]:
depth_check = pd.DataFrame([
    tune_one("LightGCN", "last_out", n_layers=1, l2_reg=reg, epochs=300)
    for reg in [3e-3, 1e-2]
])
depth_check

  LightGCN_last_out_L1_reg0.01_e300_s42: 14.1 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,LightGCN,last_out,1,0.003,42,40,0.010865,140,True
1,LightGCN,last_out,1,0.010,42,300,0.012128,300,False


In [21]:
a = phase_b.loc[phase_b["n_layers"] == 2, "val_ndcg@10"].iloc[0]
b = depth_check.loc[depth_check["l2_reg"] == 1e-2, "val_ndcg@10"].iloc[0]
print(repr(a), repr(b), a == b)

for tag in ["LightGCN_last_out_L2_reg0.003_e300_s42",
            "LightGCN_last_out_L1_reg0.01_e300_s42"]:
    h = pd.read_csv(TUNING_CACHE / f"{tag}.csv")
    print(tag, len(h), h["loss"].iloc[-1], h["seconds"].mean())

np.float64(0.012127986177802086) np.float64(0.01212847139686346) False
LightGCN_last_out_L2_reg0.003_e300_s42 300 0.0376893435194551 4.134662696520487
LightGCN_last_out_L1_reg0.01_e300_s42 300 0.0653664480608062 2.7988652761777244


In [22]:
depth_tuned = pd.DataFrame([
    tune_one("LightGCN", "last_out", n_layers=L, l2_reg=reg, epochs=300)
    for L, reg in [(1, 3e-3), (1, 1e-2), (1, 3e-2),
                   (2, 3e-3), (2, 1e-2),
                   (3, 1e-3), (3, 3e-3), (3, 1e-2),
                   (4, 1e-3), (4, 3e-3)]
])
depth_tuned.pivot(index="n_layers", columns="l2_reg", values="val_ndcg@10")

  LightGCN_last_out_L1_reg0.03_e300_s42: 12.4 min
  LightGCN_last_out_L2_reg0.01_e300_s42: 21.3 min
  LightGCN_last_out_L4_reg0.001_e300_s42: 39.0 min


l2_reg,0.001,0.003,0.010,0.030
n_layers,,,,
1,NaN,0.010865,0.012128,0.011894
2,NaN,0.012128,0.011910,NaN
3,0.012407,0.012740,0.011910,NaN
4,0.012308,0.012107,NaN,NaN


In [23]:
depth_tuned = pd.DataFrame([
    tune_one("LightGCN", "last_out", n_layers=L, l2_reg=reg, epochs=300)
    for L, reg in [(1, 3e-3), (1, 1e-2), (1, 3e-2),
                   (2, 1e-3), (2, 3e-3), (2, 1e-2),
                   (3, 1e-3), (3, 3e-3), (3, 1e-2),
                   (4, 3e-4), (4, 1e-3), (4, 3e-3)]
])
depth_tuned.pivot(index="n_layers", columns="l2_reg", values="val_ndcg@10")

  LightGCN_last_out_L2_reg0.001_e300_s42: 11.9 min
  LightGCN_last_out_L4_reg0.0003_e300_s42: 38.8 min


l2_reg,0.0003,0.0010,0.0030,0.0100,0.0300
n_layers,,,,,
1,NaN,NaN,0.010865,0.012128,0.011894
2,NaN,0.011571,0.012128,0.011910,NaN
3,NaN,0.012407,0.012740,0.011910,NaN
4,0.011678,0.012308,0.012107,NaN,NaN


In [24]:
probe_ngcf_sage = pd.DataFrame([
    tune_one(name, "last_out", n_layers=MODELS[name]["layers"], l2_reg=reg, epochs=300)
    for name in ["NGCF", "GraphSAGE"]
    for reg in [1e-3, 3e-3, 1e-2]
])
probe_ngcf_sage.pivot(index="model", columns="l2_reg", values="val_ndcg@10")

  NGCF_last_out_L3_reg0.001_e300_s42: 26.9 min
  NGCF_last_out_L3_reg0.003_e300_s42: 61.9 min
  NGCF_last_out_L3_reg0.01_e300_s42: 61.9 min
  GraphSAGE_last_out_L2_reg0.001_e300_s42: 21.4 min
  GraphSAGE_last_out_L2_reg0.003_e300_s42: 16.3 min
  GraphSAGE_last_out_L2_reg0.01_e300_s42: 24.6 min


l2_reg,0.001,0.003,0.010
model,,,
GraphSAGE,0.000976,0.000611,0.000675
NGCF,0.006302,0.006598,0.007964


In [25]:
for tag in ["GraphSAGE_last_out_L2_reg0.001_e300_s42",
            "NGCF_last_out_L3_reg0.01_e300_s42",
            "LightGCN_last_out_L3_reg0.003_e300_s42"]:
    h = pd.read_csv(TUNING_CACHE / f"{tag}.csv")
    e = h.dropna(subset=["val_ndcg@10"])
    print(f"{tag.split('_last')[0]:>10}: loss {h['loss'].iloc[0]:.4f} -> {h['loss'].iloc[-1]:.4f}"
          f" | ndcg {e['val_ndcg@10'].iloc[0]:.5f} -> {e['val_ndcg@10'].iloc[-1]:.5f}"
          f" | best epoch {int(e.loc[e['val_ndcg@10'].idxmax(), 'epoch'])}")

 GraphSAGE: loss 0.5582 -> 0.4470 | ndcg 0.00033 -> 0.00055 | best epoch 110
      NGCF: loss 0.4921 -> 0.0618 | ndcg 0.00017 -> 0.00752 | best epoch 230
  LightGCN: loss 0.6817 -> 0.0509 | ndcg 0.00999 -> 0.01274 | best epoch 300


In [28]:
sage_check = pd.DataFrame([
    tune_one("GraphSAGE", "last_out", n_layers=2, l2_reg=reg, epochs=300,
             normalise_output=False)
    for reg in [1e-3, 3e-3]
])
sage_check

  GraphSAGE_last_out_L2_reg0.001_e300_normalise_outputFalse_s42: 12.2 min
  GraphSAGE_last_out_L2_reg0.003_e300_normalise_outputFalse_s42: 14.6 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,GraphSAGE,last_out,2,0.001,42,30,0.005932,130,False
1,GraphSAGE,last_out,2,0.003,42,50,0.006032,150,False


In [29]:
ngcf_extend = pd.DataFrame([
    tune_one("NGCF", "last_out", n_layers=3, l2_reg=reg, epochs=300)
    for reg in [1e-2, 3e-2, 1e-1]
])
ngcf_extend

  NGCF_last_out_L3_reg0.03_e300_s42: 61.3 min
  NGCF_last_out_L3_reg0.1_e300_s42: 60.6 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,NGCF,last_out,3,0.01,42,230,0.007964,300,True
1,NGCF,last_out,3,0.03,42,290,0.004027,300,False
2,NGCF,last_out,3,0.10,42,250,0.003149,300,False


In [32]:
sage_grid = pd.DataFrame([
    tune_one("GraphSAGE", "last_out", n_layers=2, l2_reg=reg, epochs=300,
             normalise_output=False, final_activation=act)
    for act in [True, False]
    for reg in [1e-2, 3e-2]
])
sage_grid.pivot(index="final_activation", columns="l2_reg", values="val_ndcg@10")

l2_reg,0.01,0.03
final_activation,,
False,0.004750,0.004887
True,0.006393,0.005373


In [33]:
sage_depth = pd.DataFrame([
    tune_one("GraphSAGE", "last_out", n_layers=L, l2_reg=1e-2, epochs=300,
             normalise_output=False, final_activation=True)
    for L in [1, 2, 3]
])
sage_depth

  GraphSAGE_last_out_L1_reg0.01_e300_final_activationTrue_normalise_outputFalse_s42: 14.2 min
  GraphSAGE_last_out_L3_reg0.01_e300_final_activationTrue_normalise_outputFalse_s42: 41.8 min


,model,protocol,n_layers,l2_reg,seed,normalise_output,final_activation,best_epoch,val_ndcg@10,epochs_run,cached
0,GraphSAGE,last_out,1,0.01,42,False,True,220,0.003518,300,False
1,GraphSAGE,last_out,2,0.01,42,False,True,20,0.006393,120,True
2,GraphSAGE,last_out,3,0.01,42,False,True,230,0.006564,300,False


In [34]:
ngcf_depth = pd.DataFrame([
    tune_one("NGCF", "last_out", n_layers=L, l2_reg=1e-2, epochs=300)
    for L in [1, 2, 3, 4]
])
ngcf_depth

  NGCF_last_out_L1_reg0.01_e300_s42: 22.5 min
  NGCF_last_out_L2_reg0.01_e300_s42: 43.5 min
  NGCF_last_out_L4_reg0.01_e300_s42: 80.2 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached
0,NGCF,last_out,1,0.01,42,300,0.006877,300,False
1,NGCF,last_out,2,0.01,42,300,0.007663,300,False
2,NGCF,last_out,3,0.01,42,230,0.007964,300,True
3,NGCF,last_out,4,0.01,42,270,0.007001,300,False


In [35]:
sage_final = pd.DataFrame([
    tune_one("GraphSAGE", "last_out", n_layers=3, l2_reg=reg, epochs=300,
             normalise_output=False, final_activation=True)
    for reg in [3e-3, 1e-2, 3e-2]
])
sage_final

  GraphSAGE_last_out_L3_reg0.003_e300_final_activationTrue_normalise_outputFalse_s42: 17.9 min
  GraphSAGE_last_out_L3_reg0.03_e300_final_activationTrue_normalise_outputFalse_s42: 17.6 min


,model,protocol,n_layers,l2_reg,seed,normalise_output,final_activation,best_epoch,val_ndcg@10,epochs_run,cached
0,GraphSAGE,last_out,3,0.003,42,False,True,30,0.007793,130,False
1,GraphSAGE,last_out,3,0.010,42,False,True,230,0.006564,300,True
2,GraphSAGE,last_out,3,0.030,42,False,True,30,0.006978,130,False


In [36]:
tag = ("GraphSAGE_last_out_L3_reg0.003_e300"
       "_final_activationTrue_normalise_outputFalse_s42")
h = pd.read_csv(TUNING_CACHE / f"{tag}.csv").dropna(subset=["val_ndcg@10"])
print(h[["epoch", "loss", "val_ndcg@10"]].to_string(index=False))

 epoch     loss  val_ndcg@10
    10 0.085828     0.005689
    20 0.068832     0.007032
    30 0.059319     0.007793
    40 0.050128     0.006344
    50 0.047201     0.006271
    60 0.041822     0.007431
    70 0.039567     0.006622
    80 0.038123     0.005954
    90 0.037284     0.006359
   100 0.034440     0.005052
   110 0.032775     0.006182
   120 0.031733     0.005538
   130 0.030778     0.005615


# timestamp

In [8]:
ts_probe = tune_one("LightGCN", "timestamp", n_layers=3, l2_reg=3e-3, epochs=300)
print(ts_probe)
print(f"\nPopularity on the same evaluator: {pop_reference['timestamp']:.5f}")

  LightGCN_timestamp_L3_reg0.003_e300_s42: 37.9 min
{'model': 'LightGCN', 'protocol': 'timestamp', 'n_layers': 3, 'l2_reg': 0.003, 'seed': 42, 'best_epoch': 300, 'val_ndcg@10': 0.004904866684228182, 'epochs_run': 300, 'cached': False}

Popularity on the same evaluator: 0.00390


In [9]:
h = pd.read_csv(TUNING_CACHE / "LightGCN_timestamp_L3_reg0.003_e300_s42.csv")
e = h.dropna(subset=["val_ndcg@10"])
print(e[["epoch", "loss", "val_ndcg@10"]].tail(12).to_string(index=False))

 epoch     loss  val_ndcg@10
   190 0.063646     0.004728
   200 0.062853     0.004726
   210 0.062072     0.004849
   220 0.061553     0.004760
   230 0.061272     0.004854
   240 0.060934     0.004835
   250 0.060339     0.004783
   260 0.060275     0.004880
   270 0.059850     0.004800
   280 0.059606     0.004836
   290 0.059250     0.004889
   300 0.059102     0.004905


In [10]:
timestamp_grid = []

# LightGCN: regularisation around its leave-last-out optimum, plus a depth spot-check
for reg in [1e-3, 3e-3, 1e-2]:
    timestamp_grid.append(("LightGCN", 3, reg, {}))
for L in [2, 4]:
    timestamp_grid.append(("LightGCN", L, 3e-3, {}))

for reg in [3e-3, 1e-2, 3e-2]:
    timestamp_grid.append(("NGCF", 3, reg, {}))

for reg in [1e-3, 3e-3, 1e-2]:
    timestamp_grid.append(("GraphSAGE", 3, reg,
                           {"normalise_output": False, "final_activation": True}))

timestamp_tuning = pd.DataFrame([
    tune_one(name, "timestamp", n_layers=L, l2_reg=reg, epochs=300, **ov)
    for name, L, reg, ov in timestamp_grid
])
timestamp_tuning

  LightGCN_timestamp_L3_reg0.001_e300_s42: 36.5 min
  LightGCN_timestamp_L3_reg0.01_e300_s42: 36.2 min
  LightGCN_timestamp_L2_reg0.003_e300_s42: 26.6 min
  LightGCN_timestamp_L4_reg0.003_e300_s42: 26.0 min
  NGCF_timestamp_L3_reg0.003_e300_s42: 32.6 min
  NGCF_timestamp_L3_reg0.01_e300_s42: 63.3 min
  NGCF_timestamp_L3_reg0.03_e300_s42: 70.5 min
  GraphSAGE_timestamp_L3_reg0.001_e300_final_activationTrue_normalise_outputFalse_s42: 20.8 min
  GraphSAGE_timestamp_L3_reg0.003_e300_final_activationTrue_normalise_outputFalse_s42: 19.2 min
  GraphSAGE_timestamp_L3_reg0.01_e300_final_activationTrue_normalise_outputFalse_s42: 29.1 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached,normalise_output,final_activation
0,LightGCN,timestamp,3,0.001,42,270,0.005178,300,False,NaN,NaN
1,LightGCN,timestamp,3,0.003,42,300,0.004905,300,True,NaN,NaN
2,LightGCN,timestamp,3,0.010,42,240,0.004673,300,False,NaN,NaN
3,LightGCN,timestamp,2,0.003,42,240,0.004889,300,False,NaN,NaN
4,LightGCN,timestamp,4,0.003,42,70,0.004597,170,False,NaN,NaN
5,NGCF,timestamp,3,0.003,42,40,0.002634,140,False,NaN,NaN
6,NGCF,timestamp,3,0.010,42,170,0.003164,270,False,NaN,NaN
7,NGCF,timestamp,3,0.030,42,230,0.001765,300,False,NaN,NaN
8,GraphSAGE,timestamp,3,0.001,42,30,0.003791,130,False,False,True
9,GraphSAGE,timestamp,3,0.003,42,20,0.003568,120,False,False,True


In [11]:
boundary_check = pd.DataFrame([
    tune_one("LightGCN", "timestamp", n_layers=3, l2_reg=3e-4, epochs=300),
    tune_one("GraphSAGE", "timestamp", n_layers=3, l2_reg=3e-4, epochs=300,
             normalise_output=False, final_activation=True),
])
boundary_check

  LightGCN_timestamp_L3_reg0.0003_e300_s42: 36.7 min
  GraphSAGE_timestamp_L3_reg0.0003_e300_final_activationTrue_normalise_outputFalse_s42: 21.6 min


,model,protocol,n_layers,l2_reg,seed,best_epoch,val_ndcg@10,epochs_run,cached,normalise_output,final_activation
0,LightGCN,timestamp,3,0.0003,42,240,0.005309,300,False,NaN,NaN
1,GraphSAGE,timestamp,3,0.0003,42,30,0.004012,130,False,False,True
